# Multilingual Neuron Activation Analysis

Identifies language-specialized neurons in a causal LM by recording MLP gate activations across a set of languages, then ranking neurons by the entropy of their per-language activation distributions.

**Steps:**
1. Configure paths, model, and languages  
2. Install dependencies  
3. Download and tokenize the TED corpus  
4. Load the model and record activations  
5. Compute per-neuron language entropy and export results

## Cell 1 — Configuration

**Edit these variables before running the notebook.**

In [ ]:
# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = "jvonrad/Qwen-2.5-7B-TED-grpo"   # HuggingFace model repo (org/name)
HF_TOKEN  = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXX" # HuggingFace token (for gated models)

# ── Data ─────────────────────────────────────────────────────────────────────
# Google Drive file ID for the TED corpus zip
TED_GDRIVE_FILE_ID = "1lL0gNZAE05hZZR1sidbaUpF1JSdO6U1G"
MAX_TOKENS_PER_LANG = 2_000_000   # cap on tokens collected per language
MAX_SEQ_LENGTH      = 4_096       # token sequence length fed to the model

# ── Languages ────────────────────────────────────────────────────────────────
LANGUAGES = ['ar', 'bn', 'de', 'en', 'es', 'fr', 'id', 'ja', 'pt', 'ru', 'sw', 'zh-cn']

# ── Output ───────────────────────────────────────────────────────────────────
# Folder in Google Drive where the final JSON will be saved
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/nlp_project_research"
OUTPUT_FILENAME   = "neuron_activation_results.json"

# Share of neurons (by lowest entropy) flagged as language-specialised
TOP_SPECIALISED_FRACTION = 0.01   # 0.01 = top 1 %

# ── Internal paths (no need to change) ───────────────────────────────────────
LOCAL_DATA_DIR = "/content/data"
TED_EXTRACT_DIR = "/content/ted_data"

## Cell 2 — Mount Google Drive and create directories

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

for path in [DRIVE_RESULTS_DIR, LOCAL_DATA_DIR, TED_EXTRACT_DIR]:
    os.makedirs(path, exist_ok=True)

print("Drive mounted. Directories ready.")

## Cell 3 — Install dependencies

Upgrades the runtime to Python 3.10 and installs the required packages.  
**Expect this cell to take several minutes.**

In [ ]:
# Python 3.10
!apt-get update -q && apt-get install python3.10 python3.10-distutils python3.10-dev -q -y
!update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1
!update-alternatives --set python3 /usr/bin/python3.10
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3

# PyTorch (CUDA 12.1)
!python3 -m pip install \
    torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 \
    --index-url https://download.pytorch.org/whl/cu121

# Research stack
!python3 -m pip install \
    vllm==0.2.7 transformers datasets huggingface_hub \
    ninja accelerate "numpy<2" \
    matplotlib seaborn pandas tqdm gdown

## Cell 4 — Authenticate with HuggingFace and download the TED corpus

In [ ]:
from huggingface_hub import login

login(token=HF_TOKEN)

# Download and extract the TED corpus zip from Google Drive
zip_path = "/content/TED2025.zip"
!gdown "https://drive.google.com/uc?id={TED_GDRIVE_FILE_ID}" -O {zip_path}
!unzip -o -q {zip_path} -d {TED_EXTRACT_DIR}

print("Corpus extracted to", TED_EXTRACT_DIR)

## Cell 5 — Tokenize corpus and save token tensors

In [ ]:
import glob
import json
import torch
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Collect up to MAX_TOKENS_PER_LANG tokens for each language
lang_tokens = {lang: [] for lang in LANGUAGES}

jsonl_files = glob.glob(f"{TED_EXTRACT_DIR}/**/multi_way.jsonl", recursive=True)
print(f"Found {len(jsonl_files)} JSONL file(s).")

for jsonl_path in jsonl_files:
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            para_data = json.loads(line).get("para_data", {})
            for lang in LANGUAGES:
                text = para_data.get(lang)
                if text and len(lang_tokens[lang]) < MAX_TOKENS_PER_LANG:
                    lang_tokens[lang].extend(
                        tokenizer.encode(text, add_special_tokens=False)
                    )

# Persist token tensors to disk
for lang, tokens in lang_tokens.items():
    if tokens:
        save_path = os.path.join(LOCAL_DATA_DIR, f"tokens.{lang}.pt")
        torch.save(torch.tensor(tokens[:MAX_TOKENS_PER_LANG], dtype=torch.long), save_path)
        print(f"  {lang}: {len(tokens):,} tokens saved.")
    else:
        print(f"  {lang}: no data found — skipping.")

## Cell 6 — Load model

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print(f"Model loaded: {MODEL_ID}")
print(f"  Layers:            {model.config.num_hidden_layers}")
print(f"  Intermediate size: {model.config.intermediate_size:,}")

## Cell 7 — Record MLP gate activations

For each language, a forward-hook on every layer's `gate_proj` accumulates how often each neuron fires (activation > 0) across all token sequences.

In [ ]:
import torch.nn.functional as F
from tqdm import tqdm


def record_activations(model, lang: str) -> None:
    """Run one forward pass per sequence chunk for `lang` and save activation counts."""
    token_path = os.path.join(LOCAL_DATA_DIR, f"tokens.{lang}.pt")
    act_path   = os.path.join(LOCAL_DATA_DIR, f"activations.{lang}.pt")

    if not os.path.exists(token_path):
        print(f"  [{lang}] Token file not found — skipping.")
        return

    num_layers       = model.config.num_hidden_layers
    intermediate_size = model.config.intermediate_size

    # Integer counts: how many tokens caused neuron i in layer l to fire
    over_zero = torch.zeros(num_layers, intermediate_size, dtype=torch.int32, device="cuda")

    # Register forward hooks on every layer's gate projection
    hooks = []
    for layer_idx in range(num_layers):
        def make_hook(idx):
            def hook(module, input, output):
                gate       = output[0] if isinstance(output, tuple) else output
                activation = F.silu(gate).float()
                over_zero[idx] += (activation > 0).sum(dim=(0, 1)).to(torch.int32)
            return hook
        hooks.append(
            model.model.layers[layer_idx].mlp.gate_proj.register_forward_hook(make_hook(layer_idx))
        )

    try:
        ids = torch.load(token_path)
        # Trim to a multiple of MAX_SEQ_LENGTH
        n_tokens = (ids.size(0) // MAX_SEQ_LENGTH) * MAX_SEQ_LENGTH
        if n_tokens == 0:
            print(f"  [{lang}] Not enough tokens for a full sequence — skipping.")
            return

        sequences = ids[:n_tokens].reshape(-1, MAX_SEQ_LENGTH)

        with torch.no_grad():
            for seq in tqdm(sequences, desc=f"[{lang}]", leave=False):
                model(seq.unsqueeze(0).to("cuda"))

        torch.save({"n": n_tokens, "over_zero": over_zero.cpu()}, act_path)
        print(f"  [{lang}] Done — {n_tokens:,} tokens, saved to {act_path}")

    finally:
        for h in hooks:
            h.remove()
        torch.cuda.empty_cache()


print("Recording activations ...")
for lang in LANGUAGES:
    record_activations(model, lang)

## Cell 8 — Compute language entropy and export results

Neurons with the lowest entropy across languages are the most language-specialised. The top `TOP_SPECIALISED_FRACTION` are collected and assigned to whichever language activates them most strongly.

In [ ]:
import json

# ── Load saved activation data ────────────────────────────────────────────────
n_counts, activations, found_langs = [], [], []

for lang in LANGUAGES:
    act_path = os.path.join(LOCAL_DATA_DIR, f"activations.{lang}.pt")
    if os.path.exists(act_path):
        data = torch.load(act_path)
        n_counts.append(data["n"])
        activations.append(data["over_zero"])
        found_langs.append(lang)

if not found_langs:
    raise RuntimeError("No activation files found. Did Cell 7 complete successfully?")

print(f"Languages with activation data: {found_langs}")

# ── Build tensors ─────────────────────────────────────────────────────────────
n_tensor         = torch.tensor(n_counts)                       # (L,)
over_zero_stack  = torch.stack(activations, dim=-1)             # (layers, neurons, langs)
num_layers, intermediate_size, _ = over_zero_stack.shape

# ── Compute per-neuron entropy across languages ───────────────────────────────
activation_probs = over_zero_stack / n_tensor                   # normalise by token count
normed           = activation_probs / (activation_probs.sum(dim=-1, keepdim=True) + 1e-9)
entropy          = -torch.sum(normed * torch.log(normed + 1e-9), dim=-1)  # (layers, neurons)

# Neurons that never fire in any language get infinite entropy (not specialised)
entropy[over_zero_stack.sum(dim=-1) == 0] = float("inf")

# ── Select the most specialised neurons ──────────────────────────────────────
n_top    = round(entropy.numel() * TOP_SPECIALISED_FRACTION)
_, index = entropy.flatten().topk(n_top, largest=False)
row_idx  = index // intermediate_size   # layer index
col_idx  = index %  intermediate_size   # neuron index

# ── Assign each selected neuron to its dominant language ─────────────────────
# Keys are 1-indexed layer numbers to match typical convention
results = {layer: {lang: 0 for lang in found_langs} for layer in range(1, num_layers + 1)}

for layer, neuron in zip(row_idx, col_idx):
    dominant_lang = found_langs[activation_probs[layer, neuron].argmax().item()]
    results[layer.item() + 1][dominant_lang] += 1

# ── Save results ──────────────────────────────────────────────────────────────
output_path = os.path.join(DRIVE_RESULTS_DIR, OUTPUT_FILENAME)
with open(output_path, "w") as f:
    json.dump(results, f, indent=4)

print(f"Results saved to: {output_path}")
print(f"Total specialised neurons identified: {n_top:,} ({TOP_SPECIALISED_FRACTION:.0%} of {entropy.numel():,})")